In [23]:
from typing import List
import stim
from stimcirq import stim_circuit_to_cirq_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim

In [2]:
generators = [
    stim.PauliString("ZIZI"),
    stim.PauliString("IZIZ"),
    stim.PauliString("XIXIX"),
    stim.PauliString("IXIXIX"),
    stim.PauliString("IZIIIZZ")
]

In [3]:
for gi in generators:
    for gj in generators:
        assert gi.commutes(gj)

In [4]:
# errors = [
#     stim.PauliString("XIII"),
#     stim.PauliString("YIII"),
#     stim.PauliString("ZXII"),
#     stim.PauliString("ZYII"),
#     stim.PauliString("ZZXI"),
#     stim.PauliString("ZZYI"),
#     stim.PauliString("ZZZX"),
#     stim.PauliString("ZZZY")
# ]
errors = [
    stim.PauliString("XXII"),
    stim.PauliString("XYII"),
    stim.PauliString("YXII"),
    stim.PauliString("YYII"),
    stim.PauliString("IIXX"),
    stim.PauliString("IIXY"),
    stim.PauliString("IIYX"),
    stim.PauliString("IIYY")
]

In [5]:
number_false = 0
number_checked = 0
for i, ei in enumerate(errors):
    for j in range(i):
        number_checked += 1
        ej = errors[j]
        e = ei * ej
        commutators = []
        for generator in generators:
            comm = e.commutes(generator)
            commutators.append(comm)
        has_anticommuting_operator = any([not b for b in commutators])
        if has_anticommuting_operator:
            number_false += 1
        print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
print(f"{number_false}/{number_checked} operators anticommute.")

+XY__ * +XX__ = -i_Z__, [True, True, True, False, True] True 
+YX__ * +XX__ = -iZ___, [True, True, False, True, True] True 
+YX__ * +XY__ = +ZZ__, [True, True, False, False, True] True 
+YY__ * +XX__ = -ZZ__, [True, True, False, False, True] True 
+YY__ * +XY__ = -iZ___, [True, True, False, True, True] True 
+YY__ * +YX__ = -i_Z__, [True, True, True, False, True] True 
+__XX * +XX__ = +XXXX, [True, True, True, True, False] True 
+__XX * +XY__ = +XYXX, [True, True, True, False, False] True 
+__XX * +YX__ = +YXXX, [True, True, False, True, False] True 
+__XX * +YY__ = +YYXX, [True, True, False, False, False] True 
+__XY * +XX__ = +XXXY, [True, True, True, False, False] True 
+__XY * +XY__ = +XYXY, [True, True, True, True, False] True 
+__XY * +YX__ = +YXXY, [True, True, False, False, False] True 
+__XY * +YY__ = +YYXY, [True, True, False, True, False] True 
+__XY * +__XX = -i___Z, [True, True, True, False, True] True 
+__YX * +XX__ = +XXYX, [True, True, False, True, False] True 
+__YX * 

In [13]:
logical_ops_fermi = [
    of.FermionOperator("1^ 1"),
    of.FermionOperator("2^ 2"),
    of.FermionOperator("3^ 3"),
    of.FermionOperator("4^ 4"),
    of.FermionOperator("1^ 3"),
    of.FermionOperator("3^ 1"),
    of.FermionOperator("2^ 4"),
    of.FermionOperator("4^ 2"),
]

In [19]:
def conjugate_psum_with_circuit(psum: cirq.PauliSum, circuit: cirq.Circuit) -> cirq.PauliSum:
    new_pstrings = []
    for pstring in psum:
        new_pstrings.append(pstring.after(circuit))
    return cirq.PauliSum.from_pauli_strings(new_pstrings)

In [20]:
def conjugate_psum_with_circuit_list(psum: cirq.PauliSum, circuits: List[cirq.Circuit]) -> cirq.PauliSum:
    new_psum = psum.copy()
    for circuit in circuits:
        new_psum = conjugate_psum_with_circuit(new_psum, circuit)
    return new_psum

In [26]:
encoding_circuits = [encoding_unitary_for_new_stabilizer(gen) for gen in generators[2:]]
encoding_circuits_cirq = [stim_circuit_to_cirq_circuit(ckt) for ckt in encoding_circuits]

In [27]:
logical_ops_qubop = [of.transforms.jordan_wigner(lop) for lop in logical_ops_fermi]
logical_ops_cirq = [of.transforms.qubit_operator_to_pauli_sum(lop) for lop in logical_ops_qubop]
new_logical_ops = [conjugate_psum_with_circuit_list(lop, encoding_circuits_cirq) for lop in logical_ops_cirq]

In [29]:
for original, new in zip(logical_ops_cirq, new_logical_ops):
    print(original, '\n', new, '\n')

0.500*I-0.500*Z(q(1)) 
 0.500*I-0.500*Z(q(1))*Z(q(5)) 

0.500*I-0.500*Z(q(2)) 
 0.500*I-0.500*Z(q(2))*Z(q(4)) 

0.500*I-0.500*Z(q(3)) 
 0.500*I-0.500*Z(q(3))*Z(q(5)) 

0.500*I-0.500*Z(q(4)) 
 0.500*I-0.500*X(q(0))*X(q(2))*X(q(4)) 

-0.250j*Y(q(1))*Z(q(2))*X(q(3))+0.250*Y(q(1))*Z(q(2))*Y(q(3))+0.250*X(q(1))*Z(q(2))*X(q(3))+0.250j*X(q(1))*Z(q(2))*Y(q(3)) 
 -0.250j*Y(q(1))*Z(q(2))*X(q(3))*Z(q(4))*Z(q(5))*X(q(6))+0.250*Y(q(1))*Z(q(2))*Y(q(3))*Z(q(4))*X(q(6))+0.250*X(q(1))*Z(q(2))*X(q(3))*Z(q(4))*X(q(6))+0.250j*X(q(1))*Z(q(2))*Y(q(3))*Z(q(4))*Z(q(5))*X(q(6)) 

0.250j*Y(q(1))*Z(q(2))*X(q(3))+0.250*X(q(1))*Z(q(2))*X(q(3))+0.250*Y(q(1))*Z(q(2))*Y(q(3))-0.250j*X(q(1))*Z(q(2))*Y(q(3)) 
 0.250j*Y(q(1))*Z(q(2))*X(q(3))*Z(q(4))*Z(q(5))*X(q(6))+0.250*X(q(1))*Z(q(2))*X(q(3))*Z(q(4))*X(q(6))+0.250*Y(q(1))*Z(q(2))*Y(q(3))*Z(q(4))*X(q(6))-0.250j*X(q(1))*Z(q(2))*Y(q(3))*Z(q(4))*Z(q(5))*X(q(6)) 

-0.250j*Y(q(2))*Z(q(3))*X(q(4))+0.250*Y(q(2))*Z(q(3))*Y(q(4))+0.250*X(q(2))*Z(q(3))*X(q(4))+0.250j*X(q(2))*Z(q